In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LinearRegression

# --- CONFIGURATION & STATIC MATRIX ---
DATA_PATH = "data/FinalTransactionReport.csv"
ALPHA = 0.5  
PRICE_CEILING_MULTIPLIER = 1.5 

# Mapping the matrix to match your "Price Scale" column exactly
matrix_data = [
    {"Price Scale": "Club", "Day_Type": "Weekday", "Floor": 77.0, "Suggested": 77.0, "First3": 79.0},
    {"Price Scale": "Club", "Day_Type": "Weekend", "Floor": 77.0, "Suggested": 77.0, "First3": 79.0},
    {"Price Scale": "Diamond Box", "Day_Type": "Weekday", "Floor": 28.0, "Suggested": 24.0, "First3": 26.0},
    {"Price Scale": "Diamond Box", "Day_Type": "Weekend", "Floor": 28.0, "Suggested": 26.0, "First3": 28.0},
    {"Price Scale": "Dugout Box", "Day_Type": "Weekday", "Floor": 24.0, "Suggested": 18.0, "First3": 20.0},
    {"Price Scale": "Dugout Box", "Day_Type": "Weekend", "Floor": 24.0, "Suggested": 20.0, "First3": 22.0},
    {"Price Scale": "Field Seats", "Day_Type": "Weekday", "Floor": 18.0, "Suggested": 14.0, "First3": 16.0},
    {"Price Scale": "Field Seats", "Day_Type": "Weekend", "Floor": 18.0, "Suggested": 16.0, "First3": 18.0},
    {"Price Scale": "SRO", "Day_Type": "Weekday", "Floor": 10.0, "Suggested": 10.0, "First3": 12.0},
    {"Price Scale": "SRO", "Day_Type": "Weekend", "Floor": 10.0, "Suggested": 10.0, "First3": 12.0},
]
matrix_df = pl.DataFrame(matrix_data)

def run_pricing_pipeline():
    # 1. LOAD & PREPROCESS
    df = pl.read_csv(DATA_PATH)
    
    # Cast and clean
    df = df.with_columns([
        pl.col("Event Timestamp").str.to_datetime().alias("event_dt"),
        # Match rows 1-3 or A-C (common for 'First 3 Rows' premiums)
        pl.col("Row").cast(pl.Utf8).is_in(["1", "2", "3", "A", "B", "C"]).alias("is_front_row"),
        pl.col("Day of Week").is_in(["Friday", "Saturday", "Sunday"]).alias("is_weekend")
    ]).with_columns([
        pl.col("event_dt").dt.month().alias("month"),
    ])

    # 2. MODEL GAME-LEVEL DEMAND (Step 3)
    game_demand = df.group_by("Event ID").agg([
        pl.col("Tickets Sold").sum().alias("total_demand"),
        pl.col("is_weekend").first(),
        pl.col("month").first(),
        pl.col("Event Score").first().fill_null(50),
        pl.col("temperature_f").mean().fill_null(70)
    ])

    baseline_demand = game_demand.filter(
        (pl.col("is_weekend") == False) & (pl.col("month") == 4)
    ).select(pl.col("total_demand").mean()).item() or 1.0 # Avoid div by zero

    X = game_demand.select(["is_weekend", "month", "Event Score", "temperature_f"]).to_pandas()
    y = game_demand.select("total_demand").to_pandas()

    model = LinearRegression()
    model.fit(X, y)
    
    # Generate multiplier relative to baseline
    game_demand = game_demand.with_columns(
        (pl.lit(model.predict(X).flatten()) / baseline_demand).alias("demand_multiplier")
    )

    # 3. TRANSLATE SIGNALS TO PRICING (Step 4)
    # Tag day types to match matrix
    df = df.with_columns(
        pl.when(pl.col("is_weekend")).then(pl.lit("Weekend")).otherwise(pl.lit("Weekday")).alias("Day_Type")
    )

    # JOIN: Using 'Price Scale' as the primary key
    df = df.join(matrix_df, on=["Price Scale", "Day_Type"], how="left")

    # Safety check: are we actually matching rows?
    unmatched_count = df.filter(pl.col("Suggested").is_null()).height
    if unmatched_count > 0:
        print(f"Warning: {unmatched_count} rows did not match the price matrix. Check 'Price Scale' naming.")

    # Select base price and apply dynamic adjustment
    df = df.with_columns(
        pl.when(pl.col("is_front_row")).then(pl.col("First3")).otherwise(pl.col("Suggested")).alias("base_price")
    ).join(game_demand.select(["Event ID", "demand_multiplier"]), on="Event ID")

    # Final Price Logic with Constraints
    df = df.with_columns(
        (pl.col("base_price") * (1 + (pl.col("demand_multiplier") - 1) * ALPHA)).alias("raw_dynamic")
    ).with_columns(
        pl.min_horizontal([
            pl.max_horizontal([pl.col("raw_dynamic"), pl.col("Floor").fill_null(0)]), 
            pl.col("base_price") * PRICE_CEILING_MULTIPLIER
        ]).fill_null(pl.col("Average Unit Price")).alias("final_dynamic_price")
    )

    # 4. EVALUATE SCENARIOS (Step 5)
    evaluation = df.select([
        pl.col("Sales Total").sum().alias("Actual_Revenue"),
        (pl.col("final_dynamic_price") * pl.col("Tickets Sold")).sum().alias("Dynamic_Revenue_Projected"),
        pl.col("Tickets Sold").sum().alias("Actual_Attendance")
    ]).with_columns(
        (pl.col("Dynamic_Revenue_Projected") - pl.col("Actual_Revenue")).alias("Revenue_Delta")
    )

    print(f"\n--- Evaluation Scenarios ---")
    print(evaluation)

    # Trade-off Analysis
    # Using .mean().item() safely
    avg_hist_price = df.select(pl.col("Average Unit Price").mean()).item()
    avg_dyn_price = df.select(pl.col("final_dynamic_price").mean()).item()

    if avg_hist_price and avg_dyn_price:
        avg_price_shift = (avg_dyn_price / avg_hist_price) - 1
        print(f"\nAverage Price Shift: {avg_price_shift*100:.2f}%")
        
        # Simple elasticity estimation (PED = -1.0)
        attendance_impact = avg_price_shift * -1.0
        print(f"Estimated Attendance Change: {attendance_impact*100:.2f}%")
    else:
        print("\nError: Could not calculate price shift. Ensure Matrix and CSV names match.")

if __name__ == "__main__":
    try:
        run_pricing_pipeline()
    except Exception as e:
        import traceback
        print(f"Error executing pipeline: {e}")
        traceback.print_exc()


--- Evaluation Scenarios ---
shape: (1, 4)
┌────────────────┬───────────────────────────┬───────────────────┬───────────────┐
│ Actual_Revenue ┆ Dynamic_Revenue_Projected ┆ Actual_Attendance ┆ Revenue_Delta │
│ ---            ┆ ---                       ┆ ---               ┆ ---           │
│ f64            ┆ f64                       ┆ i64               ┆ f64           │
╞════════════════╪═══════════════════════════╪═══════════════════╪═══════════════╡
│ 967592.41      ┆ 1037236.5                 ┆ 48213             ┆ 69644.09      │
└────────────────┴───────────────────────────┴───────────────────┴───────────────┘

Average Price Shift: 10.77%
Estimated Attendance Change: -10.77%
